# Mapping and Geospatial Visualisation

## Overview

Effective geospatial visualisation communicates where patterns occur and how they relate to landscape context. Python offers a full stack from static publication maps to interactive web maps.

**Visualisation tools:**

| Tool | Output | Best for |
|---|---|---|
| GeoPandas + matplotlib | Static PNG/PDF | Publication figures |
| Folium | Interactive HTML | Web-based exploration |
| Contextily | Basemap tiles | Adding context to static maps |
| Plotly/px | Interactive charts + maps | Dashboards |
| Cartopy | Publication cartography | Projections, graticules |

**Cartographic principles:**
- Choose colour scales appropriate to the data type (sequential, diverging, qualitative)
- Always include scale bar, north arrow, and legend
- Match projection to purpose (equal-area for thematic maps; conformal for navigation)

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import geopandas as gpd
from shapely.geometry import Point, LineString, Polygon
import warnings; warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)

# Build study area datasets (BNG EPSG:27700)
sites_data = {
    'site_id':   [f'S{i:02d}' for i in range(1, 16)],
    'easting':   rng.uniform(317500, 326500, 15).astype(int),
    'northing':  rng.uniform(731500, 740500, 15).astype(int),
    'richness':  rng.integers(8, 28, 15),
    'restored':  rng.choice([True, False], 15),
    'catchment': rng.choice(['North','South','Main'], 15),
}
gdf_sites = gpd.GeoDataFrame(sites_data,
    geometry=[Point(e,n) for e,n in zip(sites_data['easting'], sites_data['northing'])],
    crs='EPSG:27700')
# Catchment polygons
polys = {
    'North': Polygon([(317000,737000),(323000,737000),(323000,741000),(317000,741000)]),
    'South': Polygon([(322000,737000),(327000,737000),(327000,741000),(322000,741000)]),
    'Main':  Polygon([(317000,731000),(327000,731000),(327000,737000),(317000,737000)]),
}
gdf_catch = gpd.GeoDataFrame({'catchment': list(polys.keys()),
    'area_km2': [p.area/1e6 for p in polys.values()]},
    geometry=list(polys.values()), crs='EPSG:27700')
print(f"Sites: {len(gdf_sites)}, Catchments: {len(gdf_catch)}")

---
## Publication-Quality Static Maps

In [ ]:
def add_scale_bar(ax, x, y, length_m, crs_unit='m', label=''):
    ax.plot([x, x+length_m], [y, y], 'k-', lw=3, transform=ax.transData)
    ax.text(x + length_m/2, y - 100, f'{label}', ha='center', va='top',
            fontsize=8, transform=ax.transData)

def add_north_arrow(ax, x, y, size=400):
    ax.annotate('N', xy=(x, y+size), xytext=(x, y),
                arrowprops=dict(arrowstyle='->', color='black', lw=2),
                fontsize=12, ha='center', va='center',
                xycoords='data', textcoords='data')

fig, ax = plt.subplots(figsize=(10,10))
# Base layer: catchments
gdf_catch.plot(ax=ax, column='catchment', cmap='Set2', alpha=0.25,
               edgecolor='#555', linewidth=1.5, legend=False)
# Site layer: size by richness, symbol by restoration status
for restored, marker, label in [(True, '^', 'Restored'), (False, 'o', 'Control')]:
    sub = gdf_sites[gdf_sites['restored'] == restored]
    sc = ax.scatter(sub.geometry.x, sub.geometry.y,
                    c=sub['richness'], cmap='YlOrRd',
                    s=sub['richness']*12, marker=marker,
                    edgecolors='black', linewidths=0.8,
                    vmin=8, vmax=28, zorder=3, label=label)
plt.colorbar(sc, ax=ax, label='Species richness', shrink=0.5, pad=0.02)
# Catchment labels
for _, row in gdf_catch.iterrows():
    cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
    ax.text(cx, cy, row['catchment'], ha='center', va='center',
            fontsize=12, color='#333', fontweight='bold', alpha=0.6)
# Map furniture
add_scale_bar(ax, 317500, 731300, 2000, label='2 km')
add_north_arrow(ax, 326500, 739500)
ax.legend(title='Site type', loc='lower right', fontsize=9)
ax.set_xlabel('Easting (m, BNG)'); ax.set_ylabel('Northing (m, BNG)')
ax.set_title('Riparian Restoration Monitoring Network\nSpecies Richness by Site Type',
             fontsize=13, pad=12)
plt.tight_layout(); plt.show()

---
## Choropleth Maps

In [ ]:
# Aggregate richness by catchment
richness_by_catch = gdf_sites.groupby('catchment')['richness'].agg(['mean','count']).reset_index()
gdf_choro = gdf_catch.merge(richness_by_catch, on='catchment')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# Sequential: mean richness
gdf_choro.plot(column='mean', ax=axes[0], cmap='YlGn', edgecolor='white',
               linewidth=1, legend=True,
               legend_kwds={'label':'Mean richness','shrink':0.7})
for _, row in gdf_choro.iterrows():
    cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
    axes[0].text(cx, cy, f"{row['mean']:.1f}", ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('Mean Richness
(Sequential scale)')
axes[0].axis('off')

# Diverging: deviation from mean
mean_all = gdf_choro['mean'].mean()
gdf_choro['deviation'] = gdf_choro['mean'] - mean_all
gdf_choro.plot(column='deviation', ax=axes[1], cmap='RdBu', edgecolor='white',
               linewidth=1, legend=True,
               legend_kwds={'label':'Deviation from mean','shrink':0.7})
for _, row in gdf_choro.iterrows():
    cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
    axes[1].text(cx, cy, f"{row['deviation']:+.1f}", ha='center', fontsize=12, fontweight='bold')
axes[1].set_title('Deviation from Mean\n(Diverging scale)')
axes[1].axis('off')

# Categorical: restoration coverage
gdf_choro['dominant_type'] = gdf_sites.groupby('catchment')['restored'].apply(
    lambda x: 'Mostly restored' if x.mean() > 0.5 else 'Mostly control').reset_index(drop=True).values
gdf_choro.plot(column='dominant_type', ax=axes[2], cmap='Set1',
               edgecolor='white', linewidth=1, legend=True)
axes[2].set_title('Dominant Site Type
(Qualitative scale)'); axes[2].axis('off')
plt.suptitle('Choropleth Maps: Scale Type Selection'); plt.tight_layout(); plt.show()

---
## Interactive Maps with Folium

In [ ]:
try:
    import folium
    # Reproject to WGS84 for web mapping
    gdf_wgs = gdf_sites.to_crs('EPSG:4326')
    gdf_catch_wgs = gdf_catch.to_crs('EPSG:4326')
    centre = [gdf_wgs.geometry.y.mean(), gdf_wgs.geometry.x.mean()]
    m = folium.Map(location=centre, zoom_start=12, tiles='CartoDB positron')
    # Add catchment polygons
    folium.GeoJson(gdf_catch_wgs.__geo_interface__,
        style_function=lambda f: {'fillColor':'#3498db','color':'grey',
                                   'weight':1,'fillOpacity':0.15}).add_to(m)
    # Add monitoring sites as circle markers
    cmap_rich = plt.cm.YlOrRd
    norm_rich  = mcolors.Normalize(8, 28)
    for _, row in gdf_wgs.iterrows():
        hex_color = mcolors.to_hex(cmap_rich(norm_rich(row['richness'])))
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=8,
            color='black', weight=1,
            fill=True, fill_color=hex_color, fill_opacity=0.85,
            popup=folium.Popup(
                f"<b>{row['site_id']}</b><br>Richness: {row['richness']}<br>"
                f"Restored: {row['restored']}", max_width=200)
        ).add_to(m)
    # Save
    import tempfile, os
    tmp = os.path.join(tempfile.gettempdir(), 'monitoring_map.html')
    m.save(tmp)
    print(f"Interactive map saved to: {tmp}")
    print("Open in browser to explore clickable site markers")
except ImportError:
    print("pip install folium  to create interactive web maps")
    print("Folium creates Leaflet.js maps with clickable popups, layer control,")
    print("and multiple basemap options (OpenStreetMap, CartoDB, Stamen)")

In [ ]:
# Multi-panel figure: small multiples by catchment
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
catchments = ['North','South','Main']
for ax, catch_name in zip(axes, catchments):
    catch_poly = gdf_catch[gdf_catch['catchment']==catch_name]
    catch_sites = gdf_sites[gdf_sites['catchment']==catch_name]
    catch_poly.plot(ax=ax, color='#ecf0f1', edgecolor='#555', linewidth=1.5)
    if len(catch_sites):
        sc = catch_sites.plot(ax=ax, column='richness', cmap='YlOrRd',
                               markersize=100, edgecolor='black', linewidth=0.5,
                               vmin=8, vmax=28, legend=False, zorder=3)
    for _, row in catch_sites.iterrows():
        ax.annotate(str(row['richness']),
                    (row.geometry.x+100, row.geometry.y+100), fontsize=8)
    ax.set_title(f"{catch_name} catchment
n={len(catch_sites)} sites, "
                 f"mean richness={catch_sites['richness'].mean():.1f}")
    ax.axis('off')
# Shared colourbar
sm = plt.cm.ScalarMappable(cmap='YlOrRd', norm=plt.Normalize(vmin=8, vmax=28))
fig.colorbar(sm, ax=axes, label='Species richness', shrink=0.6, pad=0.02)
plt.suptitle('Small Multiples: Richness by Catchment', fontsize=13)
plt.tight_layout(); plt.show()

---

## Common Pitfalls

**1. Using a rainbow (jet) colour scale for continuous data**  
Rainbow colour scales (jet, hsv) have no perceptual ordering — equal steps in data value produce unequal perceptual differences, creating false visual boundaries. Use perceptually uniform sequential scales (viridis, plasma, YlOrRd) for continuous data, diverging scales (RdBu, PuOr) for data with a meaningful midpoint, and qualitative scales (Set1, Set2) for categories.

**2. Choosing choropleth class boundaries without checking the data distribution**  
Equal-interval classification ignores data distribution and can create classes with very unequal numbers of observations. Always inspect the histogram before choosing classification — use quantile breaks for skewed distributions, natural breaks (Jenks) for data with natural clusters, and equal-interval only for uniformly distributed data.

**3. Reprojecting to WGS84 for area or distance calculations before mapping**  
For web mapping with Folium or Leaflet, WGS84 (EPSG:4326) is required. But if you also need to compute areas, distances, or buffers, always do those calculations in a metric CRS first, then reproject only for the final visualisation step.

**4. Not specifying `vmin` and `vmax` when making small-multiples maps**  
When creating multiple map panels for comparison, each panel defaults to its own colour scale range. This makes visual comparison between panels misleading — the same colour means different values. Always set identical `vmin` and `vmax` across all panels and use a single shared colourbar.

**5. Omitting map furniture (scale bar, north arrow, legend) from publication figures**  
A map without a scale bar forces readers to guess distances. A map without a north arrow assumes all readers know the orientation. A map without a legend is uninterpretable. All three are mandatory for any map intended for communication or publication — matplotlib provides tools to add them, or use `matplotlib_scalebar` for clean scale bars.
---
*python_methods_library - Samantha McGarrigle*